# GravicodeScience on streams: DataFrames, clustering, novelty
**EN** — Turn a window of checkouts into a GraviFrame DataFrame, segment customers with GraviLearn K-Means and catch unusual checkouts with a one-class SVM.

**ID** — Ubah jendela transaksi menjadi DataFrame GraviFrame, segmentasikan pelanggan dengan K-Means GraviLearn, dan tangkap transaksi tidak wajar dengan one-class SVM.

*Dibuat oleh Gravicode Studios dipimpin oleh Kang Fadhil.*

In [ ]:
// Using a local build instead of nuget.org? Uncomment and point to <repo>/artifacts/nuget (absolute path):
// #i "nuget: C:/src/bigpipe/artifacts/nuget"
#r "nuget: BigPipe.Analytics.Gravicode, 0.1.0"
using BigPipe.Client;
using BigPipe.Client.Admin;
using BigPipe.Analytics;
var bootstrap = Environment.GetEnvironmentVariable("BIGPIPE_BOOTSTRAP") ?? "localhost:9092";
var admin = new BigPipeAdminClient(Environment.GetEnvironmentVariable("BIGPIPE_ADMIN") ?? "http://localhost:9644");
var http = new BigPipeHttpClient(Environment.GetEnvironmentVariable("BIGPIPE_HTTP") ?? "http://localhost:8082");
string Unique(string p) => $"nb-{p}-{Guid.NewGuid().ToString("N")[..6]}";
var cluster = await admin.GetClusterAsync();
Console.WriteLine($"Connected to {cluster.ClusterId} (BigPipe {cluster.Version}) — {cluster.Credit}");

In [ ]:
using BigPipe.Analytics.Gravicode;
var topic = Unique("checkouts");
await admin.CreateTopicAsync(topic, 1);
var rng = new Random(4);
var cities = new[] { "Jakarta", "Bandung", "Surabaya" };
await http.ProduceAsync(topic, Enumerable.Range(0, 600).Select(i =>
{
    var kind = i % 3;
    var basket = kind switch { 0 => rng.Next(20, 80), 1 => rng.Next(150, 400), _ => rng.Next(1500, 4000) } * 1000;
    var items = kind == 1 ? rng.Next(4, 9) : rng.Next(1, 3);
    return new HttpProduceRecord { Key = $"cust-{i}", Value = new { basket, items, city = cities[rng.Next(3)] } };
}));
var window = await BigPipeSource.Kafka(bootstrap, topic, from: OffsetReset.Earliest).CollectAsync(600);
var frame = window.ToDataFrame(["basket", "items"], ["city"]);
frame.Describe().ToString()

In [ ]:
frame.GroupBy("city").Mean("basket", "items").ToString()

In [ ]:
var seg = new GraviKMeansSegmenter(clusters: 3);
var rows = window.Select(r => new[] { r.Num("basket") / 100_000, r.Num("items") }).ToList();
seg.Fit(rows);
seg.AssignAll(rows).Select((label, i) => (label, i)).GroupBy(x => x.label)
   .Select(g => new { segment = g.Key, customers = g.Count(), avgBasket = g.Average(x => window[x.i].Num("basket")).ToString("N0") })

In [ ]:
var novelty = new GraviOneClassSvmDetector(nu: 0.02);
novelty.Fit(rows);
new[] { new[] { 0.5, 1.0 }, new[] { 2.5, 6.0 }, new[] { 120.0, 40.0 } }
    .Select(x => new { basket = x[0] * 100_000, items = x[1], unusual = novelty.Observe(x).IsAnomaly })